# Kumasi Commute-Time Prediction (Multiple Linear Regression)
**Problem:** How long (in minutes) does a trip from a suburb of Kumasi to the Central Business District (Kejetia / Adum) take?

Factors considered: distance, junctions, time of day, weekday/weekend, weather, transport mode (trotro, taxi, private car, okada), road condition, roadworks and Kejetia market days.

**Method:** same workflow as the house-pricing notebook: EDA, dummy variables, train/test split, MinMax scaling, RFE feature selection, OLS model building with p-values and VIF, residual analysis and evaluation with R-squared.

> Note: the dataset is **synthetic** (generated in Step 2 from realistic assumptions about Kumasi traffic). It is not real survey data.

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## Step 2: Create the synthetic Kumasi dataset

In [ ]:
np.random.seed(42)          # fixed seed so everyone gets the same dataset
n = 1500                    # number of trips (rows) to simulate

# Typical road distance (km) from each suburb to the CBD (Kejetia/Adum)
areas = {'Adum':1.5,'Bantama':3.5,'Suame':5,'Asokwa':4,'Ahodwo':5,'Nhyiaeso':4.5,'Santasi':7,'Kwadaso':8,
         'Atonsu':6,'Ayigya':8,'KNUST':7.5,'Oforikrom':9,'Tafo':8,'Abuakwa':12,'Kentinkrono':13,'Ejisu':20}

# ---- Input features (what we know before the trip) ----
origin = np.random.choice(list(areas), n)
dist = np.array([areas[a] for a in origin]) * np.random.uniform(0.85, 1.2, n)   # route varies a little
junctions = np.random.poisson(dist/2.5 + 1)                                      # longer trip = more junctions
tod = np.random.choice(['morning_peak','midday','evening_peak','night'], n, p=[.3,.3,.3,.1])
day = np.random.choice(['weekday','weekend'], n, p=[.7,.3])
weather = np.random.choice(['dry','rainy'], n, p=[.7,.3])
mode = np.random.choice(['trotro','taxi','private_car','okada'], n, p=[.4,.2,.25,.15])
road = np.random.choice(['good','fair','poor'], n, p=[.3,.45,.25])
roadworks = np.random.choice(['yes','no'], n, p=[.2,.8])
market = np.random.choice(['yes','no'], n, p=[.35,.65])

# ---- Hidden "real world" rule that produces the travel time ----
speed = pd.Series(mode).map({'trotro':18,'taxi':24,'private_car':26,'okada':32}).values.astype(float)  # km/h
speed *= pd.Series(tod).map({'morning_peak':.55,'midday':.8,'evening_peak':.5,'night':1.1}).values
speed *= np.where(day=='weekend', 1.15, 1.0) * np.where(weather=='rainy', .8, 1.0)
speed *= pd.Series(road).map({'good':1.0,'fair':.9,'poor':.75}).values
speed *= np.where(roadworks=='yes', .8, 1.0) * np.where(market=='yes', .9, 1.0)
wait = pd.Series(mode).map({'trotro':8,'taxi':3,'private_car':0,'okada':2}).values   # waiting/loading time (min)
t = dist/speed*60 + junctions*0.7 + wait            # travel time in minutes
t = t * np.random.lognormal(0, 0.12, n)             # random real-life noise

df = pd.DataFrame(dict(origin_area=origin, distance_km=dist.round(1), num_junctions=junctions,
                       time_of_day=tod, day_type=day, weather=weather, transport_mode=mode,
                       road_condition=road, roadworks=roadworks, market_day=market,
                       travel_time_min=t.round(1)))
df.to_csv('kumasi_commute_time.csv', index=False)   # save so it can be shared / reloaded
print('Saved kumasi_commute_time.csv', df.shape)

## Step 3: Read the data

In [ ]:
df = pd.read_csv('kumasi_commute_time.csv')

**Display some rows**

In [ ]:
df.head()

**Number of rows and columns**

In [ ]:
df.shape

**Summary statistics**

In [ ]:
df.describe()

In [ ]:
df.info()

**Duplicate rows?**

In [ ]:
df.loc[df.duplicated()]

**Column names**

In [ ]:
df.columns

**Missing values?**

In [ ]:
df.isna().sum()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(20,8))

plt.subplot(1,2,1)
plt.title('Travel Time Distribution Plot')
sns.histplot(df.travel_time_min, kde=True)

plt.subplot(1,2,2)
sns.boxplot(x=df.travel_time_min)
plt.title('Travel Time Spread')

plt.show()

In [ ]:
print(df.travel_time_min.describe(percentiles=[0.25,0.50,0.75,0.85,0.90,1]))

**Inference:** travel time is right-skewed: most trips take under about 1 hour but a few very long trips (heavy peak traffic from far suburbs) pull the mean above the median.

**Categorical variables**

In [ ]:
categorical_list = [x for x in df.columns if df[x].dtype == 'object' or str(df[x].dtype).startswith('str')]
print(categorical_list)

In [ ]:
plt.figure(figsize=(18,10))
for i, col in enumerate(['time_of_day','transport_mode','road_condition','weather','day_type','market_day'], 1):
    plt.subplot(2,3,i)
    sns.boxplot(x=df[col], y=df['travel_time_min'], palette='cubehelix')
    plt.title(col + ' vs Travel Time')
plt.tight_layout()
plt.show()

**Inference:** peak hours, trotro, poor roads, rain and market days all push travel time up; night trips and weekends are faster.

In [ ]:
plt.figure(figsize=(14,5))
order = df.groupby('origin_area')['travel_time_min'].mean().sort_values().index
sns.barplot(x='origin_area', y='travel_time_min', data=df, order=order, palette='viridis')
plt.xticks(rotation=60)
plt.title('Average travel time to the CBD by suburb')
plt.show()

**Numerical variables**

In [ ]:
numerical_list = [x for x in df.columns if df[x].dtype in ('int64','float64')]
print(numerical_list)

In [ ]:
def scatter(x, fig):
    plt.subplot(1,2,fig)
    plt.scatter(df[x], df['travel_time_min'], alpha=0.4)
    plt.title(x + ' vs Travel Time')
    plt.ylabel('Travel time (min)')
    plt.xlabel(x)

plt.figure(figsize=(12,5))
scatter('distance_km', 1)
scatter('num_junctions', 2)
plt.tight_layout()

**Inference:** distance has a strong positive relationship with travel time.

In [ ]:
sns.pairplot(df[numerical_list])
plt.show()

In [ ]:
cor_matrix = df[numerical_list].corr()
plt.figure(figsize=(8,5))
sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## Step 5: Dummy variables
Regression needs numbers, so every category becomes a 0/1 column. `drop_first=True` removes one category per variable to avoid perfect collinearity (the dummy-variable trap).

In [ ]:
# Convert all text columns to 0/1 dummy columns (prefix keeps names unique, e.g. weather_rainy)
df = pd.get_dummies(df, columns=categorical_list, drop_first=True, dtype=float)

In [ ]:
df.tail()

In [ ]:
df.shape

## Step 6: Train / test split and scaling

In [ ]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
df_train, df_test = train_test_split(df, train_size=0.75, test_size=0.25, random_state=100)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_train[numerical_list] = scaler.fit_transform(df_train[numerical_list])   # learn min/max from TRAIN only
df_test[numerical_list] = scaler.transform(df_test[numerical_list])         # reuse the same min/max on TEST

In [ ]:
df_train.head()

In [ ]:
# Divide into X (inputs) and y (target)
y_train = df_train.pop('travel_time_min')
X_train = df_train

## Step 7: Feature selection with RFE

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
rfe = RFE(estimator=LinearRegression(), n_features_to_select=12)
rfe = rfe.fit(X_train, y_train)

In [ ]:
list(zip(X_train.columns, rfe.support_, rfe.ranking_))

In [ ]:
X_train.columns[rfe.support_]

In [ ]:
X_train_rfe = X_train[X_train.columns[rfe.support_]]
X_train_rfe.head()

## Step 8: Build the model (statsmodels OLS) with helper functions

In [ ]:
def build_model(X, y):
    X = sm.add_constant(X)        # add intercept
    lm = sm.OLS(y, X).fit()       # fit ordinary least squares
    print(lm.summary())           # coefficients, p-values, R-squared
    return X

def checkVIF(X):
    vif = pd.DataFrame()
    vif['Features'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif['VIF'] = round(vif['VIF'], 2)
    vif = vif.sort_values(by="VIF", ascending=False)
    return vif

**MODEL 1**

In [ ]:
X_train_new = build_model(X_train_rfe, y_train)

All p-values are below 0.05, but `origin_area_Ejisu` is the weakest predictor (largest p-value) and it carries the same information as `distance_km` (Ejisu is simply the farthest suburb). Check VIF to confirm redundancy.

In [ ]:
checkVIF(X_train_new.drop('const', axis=1))

Dropping `origin_area_Ejisu` to keep the model simple and avoid redundancy.

In [ ]:
X_train_new = X_train_new.drop(['origin_area_Ejisu'], axis=1)

**MODEL 2**

In [ ]:
X_train_new = build_model(X_train_new, y_train)

In [ ]:
checkVIF(X_train_new.drop('const', axis=1))

All VIF values are below 5, so there is no serious multicollinearity. The model is ready.

## Step 9: Residual analysis

In [ ]:
lm = sm.OLS(y_train, X_train_new).fit()
y_train_pred = lm.predict(X_train_new)

In [ ]:
fig = plt.figure()
sns.histplot((y_train - y_train_pred), bins=20, kde=True)
fig.suptitle('Error Terms', fontsize=20)
plt.xlabel('Errors', fontsize=18)

Error terms are roughly bell-shaped and centred on zero, so the normality assumption of linear regression is approximately satisfied (a mild right tail remains because the real process is multiplicative).

## Step 10: Predict on the test set

In [ ]:
y_test = df_test.pop('travel_time_min')
X_test = df_test

In [ ]:
X_train_new = X_train_new.drop('const', axis=1)
X_test_new = X_test[X_train_new.columns]       # keep only the chosen features
X_test_new = sm.add_constant(X_test_new)

In [ ]:
y_pred = lm.predict(X_test_new)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
print('Test R-squared:', r2_score(y_test, y_pred))

# Convert back to real minutes to get an easy-to-explain error
t_min, t_max = scaler.data_min_[numerical_list.index('travel_time_min')], scaler.data_max_[numerical_list.index('travel_time_min')]
mae_minutes = mean_absolute_error(y_test, y_pred) * (t_max - t_min)
print('Mean absolute error: about', round(mae_minutes, 1), 'minutes')

In [ ]:
fig = plt.figure()
plt.scatter(y_test, y_pred, alpha=0.5)
fig.suptitle('y_test vs y_pred', fontsize=20)
plt.xlabel('y_test', fontsize=18)
plt.ylabel('y_pred', fontsize=16)

In [ ]:
print(lm.summary())

## Step 11: Use the model for a new trip

In [ ]:
def predict_trip(**trip):
    """Give the chosen features (scaled 0-1 for numeric ones) and get travel time in minutes."""
    row = pd.DataFrame([{c: 0.0 for c in X_train_new.columns}])
    for k, v in trip.items():
        row[k] = v
    row = sm.add_constant(row, has_constant='add')
    scaled = lm.predict(row)[0]
    return scaled * (t_max - t_min) + t_min

# Example: 8 km trip, trotro, rainy, morning peak
d_min, d_max = scaler.data_min_[numerical_list.index('distance_km')], scaler.data_max_[numerical_list.index('distance_km')]
j_min, j_max = scaler.data_min_[numerical_list.index('num_junctions')], scaler.data_max_[numerical_list.index('num_junctions')]
trip = dict(distance_km=(8-d_min)/(d_max-d_min), num_junctions=(5-j_min)/(j_max-j_min),
            weather_rainy=1, transport_mode_trotro=1, time_of_day_morning_peak=1)
print('Estimated travel time:', round(predict_trip(**trip), 1), 'minutes')

## Conclusion
- The model explains a large share of the variation in Kumasi commute time (see R-squared above).
- Distance is the strongest driver; trotro use, peak hours, rain, poor roads, roadworks and market days add delay.
- Limitation: the data is synthetic, so a real survey (GPS traces or commuter questionnaires) would be needed before using this for real decisions.